# Data stats

In [1]:
import pandas as pd
import numpy as np

## Readmission

In [2]:
mimicdata = pd.read_parquet('gdata.parquet')

In [4]:
mimicdata.admission_id.nunique()

82480

In [5]:
mimicdata_ = pd.read_parquet('../Dataset/mimic-iv/from_pipeline/ltm_grided_clipped.parquet')

In [7]:
mimicdata_.columns

Index(['stay_id', 'grid_end', 'vent_mode__hours_since_last__last_12h',
       'temperature__mean__last_12h', 'heart_rate__mean__last_12h',
       'arterial_blood_pressure_mean__mean__last_12h',
       'fluid_out_urine__mean__last_12h', 'pco2_arterial__mean__last_12h',
       'respiratory_rate_measured__mean__last_12h',
       'o2_saturation__mean__last_12h', 'po2_arterial__mean__last_12h',
       'bicarbonate_arterial__last__last_12h',
       'activated_partial_thromboplastin_time__last__last_12h',
       'hemoglobin__last__last_12h', 'creatinine__last__last_12h',
       'ureum__last__last_12h', 'lactate__last__last_12h',
       'glasgow_coma_scale_total__last__last_12h', 'o2_flow__last__last_12h',
       'vent_mode__last__last_12h', 'raw_age', 'sex', 'raw_height',
       'raw_weight', 'unit_type', 'origin', 'los', 'intime', 'outtime',
       'death_time_from_intime', 'icu_mortality', 'death_abs_time',
       'mortality_after_discharge', 'icu_mortality_derived'],
      dtype='object')

In [17]:
mimicdata.los

0               1 days 15:17:10
1               1 days 15:17:10
2               1 days 15:14:48
3               1 days 15:14:48
4        2 days 11:29:35.999999
                  ...          
599096          5 days 20:07:00
599097          5 days 20:07:00
599098          5 days 20:07:00
599099          5 days 20:07:00
599100          5 days 20:07:00
Name: los, Length: 574000, dtype: timedelta64[ns]

# Stats

In [20]:
# ================================
# BASIC COUNTS
# ================================
n_patients = mimicdata['subject_id'].nunique()
n_admissions = mimicdata['admission_id'].nunique()

print(f"Unique patients: {n_patients}")
print(f"Unique ICU admissions: {n_admissions}")

# ================================
# READMISSION STATISTICS
# ================================
adm_per_patient = mimicdata.groupby('subject_id')['admission_id'].nunique()

n_single_adm = (adm_per_patient == 1).sum()
n_multi_adm = (adm_per_patient > 1).sum()
prop_multi = n_multi_adm / len(adm_per_patient)

print("\nReadmission statistics:")
print(f"Patients with 1 admission: {n_single_adm}")
print(f"Patients with >1 admission: {n_multi_adm}")
print(f"Proportion with readmission: {prop_multi:.3f}")

# Optional: distribution summary
print("\nAdmissions per patient summary:")
print(adm_per_patient.describe())

# ================================
# LENGTH OF STAY (LOS)
# ================================
# Ensure one row per ICU admission
los_td = mimicdata[['admission_id', 'los']].drop_duplicates()['los']

# Convert to days (float)
los_days = los_td.dt.total_seconds() / (24 * 60 * 60)

# ================================
# LOS Summary 
# ================================
# ================================
# Truncated at K = 90 days
# ================================
los_trunc = np.minimum(los_days, 90)

print("\nLOS truncated at 90 days (K=90 follow-up):")
print(f"Mean truncated LOS: {los_trunc.mean():.2f}")
print(f"Median truncated LOS: {los_trunc.median():.2f}")
print(f"IQR truncated LOS: {np.quantile(los_trunc,0.25):.2f} - {np.quantile(los_trunc,0.75):.2f}")

# ================================
# Proportion exceeding key thresholds
# ================================
print("\nProportion exceeding thresholds:")
print(f">7 days: {(los_days > 7).mean():.3f}")
print(f">14 days: {(los_days > 14).mean():.3f}")
print(f">30 days: {(los_days > 30).mean():.3f}")
print(f">90 days: {(los_days > 90).mean():.3f}")

# ================================
# SEX DISTRIBUTION
# ================================
sex_dist = mimicdata[['subject_id','sex']].drop_duplicates()['sex'].value_counts()
sex_prop = sex_dist / sex_dist.sum()

print("\nSex distribution:")
for s in sex_dist.index:
    print(f"{s}: {sex_dist[s]} ({sex_prop[s]:.3f})")

# ================================
# AGE DISTRIBUTION
# ================================
age = mimicdata[['subject_id','age']].drop_duplicates()['age']

print("\nAge summary (years):")
print(f"Mean age: {age.mean():.2f}")
print(f"Median age: {age.median():.2f}")
print(f"IQR age: {age.quantile(0.25):.1f} - {age.quantile(0.75):.1f}")
print(f"Min–Max age: {age.min():.1f} - {age.max():.1f}")

# Optional categorical age bands
age_bins = [18, 30, 40, 50, 60, 70, 80, 90, 120]
age_groups = pd.cut(age, bins=age_bins)
print("\nAge group distribution:")
print(age_groups.value_counts().sort_index())

# ================================
# ICU ORIGIN DISTRIBUTION
# ================================
origin_dist = mimicdata[['admission_id','origin']].drop_duplicates()['origin'].value_counts()
origin_prop = origin_dist / origin_dist.sum()

print("\nICU origin distribution:")
for o in origin_dist.index:
    print(f"{o}: {origin_dist[o]} ({origin_prop[o]:.3f})")

Unique patients: 60616
Unique ICU admissions: 82480

Readmission statistics:
Patients with 1 admission: 47850
Patients with >1 admission: 12766
Proportion with readmission: 0.211

Admissions per patient summary:
count    60616.000000
mean         1.360697
std          1.021664
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         37.000000
Name: admission_id, dtype: float64

LOS truncated at 90 days (K=90 follow-up):
Mean truncated LOS: 3.93
Median truncated LOS: 2.14
IQR truncated LOS: 1.27 - 4.11

Proportion exceeding thresholds:
>7 days: 0.130
>14 days: 0.045
>30 days: 0.008
>90 days: 0.000

Sex distribution:
1.0: 34197 (0.564)
0.0: 26419 (0.436)

Age summary (years):
Mean age: 63.61
Median age: 65.00
IQR age: 54.0 - 76.0
Min–Max age: 18.0 - 91.0

Age group distribution:
age
(18, 30]      2967
(30, 40]      3369
(40, 50]      5885
(50, 60]     11195
(60, 70]     14340
(70, 80]     12609
(80, 90]      7777
(90, 120]     2383
Name: count, 

In [21]:
last_rows = (
        mimicdata.sort_values(['admission_id', 't0'])
          .groupby('admission_id')
          .tail(1)
    )
    
D_mean = last_rows['D'].fillna(0).mean()
Z_mean = last_rows['Z'].fillna(0).mean()
Y_mean = last_rows['Y'].mean()

results = []
label = "MIMIC"
results.append({
    "Strategy": label,
    "Mean D (stay-level)": D_mean,
    "Mean Z (stay-level)": Z_mean,
    "Mean Y (stay-level)": Y_mean
})
summary_table = pd.DataFrame(results)
summary_table

,Strategy,Mean D (stay-level),Mean Z (stay-level),Mean Y (stay-level)
0,MIMIC,0.073824,0.126988,0.200812


In [22]:
print("in-icu mortality", 7.4)
print("post-discharge mortality", 12.7)

in-icu mortality 7.4
post-discharge mortality 12.7


In [8]:
mimicdata__ = pd.read_parquet("../Dataset/mimic-iv/from_pipeline/ltm.parquet")

In [12]:
num_subjects_multiple_stays = (
    mimicdata__.groupby("subject_id")["stay_id"]
    .nunique()
    .gt(1)
    .sum()
)

In [16]:
num_subjects_multiple_stays/mimicdata__.stay_id.nunique()

np.float64(0.1719208327597344)

In [5]:
mimicdata.columns

Index(['admission_id', 'grid_end', 'vent_mode__hours_since_last__last_12h',
       'temperature__mean__last_12h', 'heart_rate__mean__last_12h',
       'arterial_blood_pressure_mean__mean__last_12h',
       'fluid_out_urine__mean__last_12h', 'pco2_arterial__mean__last_12h',
       'respiratory_rate_measured__mean__last_12h',
       'o2_saturation__mean__last_12h', 'po2_arterial__mean__last_12h',
       'bicarbonate_arterial__last__last_12h',
       'activated_partial_thromboplastin_time__last__last_12h',
       'hemoglobin__last__last_12h', 'creatinine__last__last_12h',
       'ureum__last__last_12h', 'lactate__last__last_12h',
       'glasgow_coma_scale_total__last__last_12h', 'o2_flow__last__last_12h',
       'vent_mode__last__last_12h', 'subject_id', 'age', 'sex', 'height',
       'weight', 'unit_type', 'origin', 'los', 'intime', 'outtime',
       'death_time_from_intime', 'icu_mortality_flag', 'death_abs_time',
       'mortality_after_discharge', 'icu_mortality', 't0', 'ref_time',
 

In [6]:
id_col = "admission_id"
time_col = "t0"

# Number of rows and admissions
n_rows = len(mimicdata)
n_adm = mimicdata[id_col].nunique()

# Last row per admission
last = (
    mimicdata.sort_values([id_col, time_col])
              .groupby(id_col)
              .tail(1)
              .reset_index(drop=True)
)

# Follow-up length (discrete time bins)
los = (
    mimicdata.groupby(id_col)[time_col]
              .max()
              .rename("max_t")
)

median_los = los.median()
iqr_los = los.quantile([0.25, 0.75]).values
max_los = los.max()

print("Rows:", n_rows)
print("Unique admissions:", n_adm)
print("Median LOS (t0 bins):", median_los)
print("IQR LOS:", iqr_los)
print("Max LOS:", max_los)


Rows: 574000
Unique admissions: 82480
Median LOS (t0 bins): 2.0
IQR LOS: [1. 6.]
Max LOS: 180


In [8]:
# In-ICU mortality
icu_mort = last["D"].mean()

# Post-discharge mortality
post_mort = last["Z"].mean()

# Composite mortality
comp_mort = last["Y"].mean()

# Discharge proportion
discharged = last["A"].mean()

print("In-ICU mortality:", icu_mort)
print("Post-discharge mortality:", post_mort)
print("Composite mortality:", comp_mort)
print("Discharged:", discharged)


In-ICU mortality: 0.07374541718235528
Post-discharge mortality: 0.13216028769586438
Composite mortality: 0.20590570487821966
Discharged: 0.9260911192583425


In [9]:
baseline = mimicdata[mimicdata[time_col] == 0]

# Example numeric summaries
numeric_cols = baseline.select_dtypes(include=[np.number]).columns.tolist()

baseline_summary = baseline[numeric_cols].agg(
    ["mean", "std", "median", "min", "max"]
).T

print(baseline_summary.head())


                                                         mean  \
grid_end                                      0 days 12:00:00   
vent_mode__hours_since_last__last_12h                3.448964   
temperature__mean__last_12h                         36.821391   
heart_rate__mean__last_12h                          85.290085   
arterial_blood_pressure_mean__mean__last_12h        79.609515   

                                                          std  \
grid_end                                      0 days 00:00:00   
vent_mode__hours_since_last__last_12h                2.888592   
temperature__mean__last_12h                          0.511776   
heart_rate__mean__last_12h                          16.940082   
arterial_blood_pressure_mean__mean__last_12h        12.256387   

                                                       median  \
grid_end                                      0 days 12:00:00   
vent_mode__hours_since_last__last_12h                2.633333   
temperature__mean__last

In [10]:
# Time to discharge
time_to_discharge = (
    mimicdata[mimicdata["A"] == 1]
    .groupby(id_col)[time_col]
    .min()
)

print("Median time to discharge:", time_to_discharge.median())

# Time to ICU death
time_to_icu_death = (
    mimicdata[mimicdata["D"] == 1]
    .groupby(id_col)[time_col]
    .min()
)

print("Median time to ICU death:", time_to_icu_death.median())


Median time to discharge: 2.0
Median time to ICU death: 6.0
